In [6]:
import pandas as pd

STEP1: Cleaning and Formatting The Financial Report

In [13]:
import pandas as pd

file_path = "C:/Users/user/Desktop/Deutsche_Bank_NetRevenue_Report.xlsx"

def clean_excel_sheet(file_path, sheet_name, quarters):
    df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=2)

    df.columns = [str(c).strip() for c in df.iloc[0]]
    df = df.iloc[1:].reset_index(drop=True)
    df.rename(columns={df.columns[0]: 'segment'}, inplace=True)
    cols_to_keep = ['segment'] + quarters
    df_filtered = df[cols_to_keep].dropna(subset=['segment']).copy()

    # Unpivot / Melt wide columns into long database format
    df_long = df_filtered.melt(
        id_vars=['segment'],
        value_vars=quarters,
        var_name='period',
        value_name='revenue_eur_m'
    )

    # Ensure revenue values are float
    df_long['revenue_eur_m'] = pd.to_numeric(df_long['revenue_eur_m'], errors='coerce')
    return df_long

# Define Quarters for Sheet 1 (Starts from Q1 2023)
q_22_24 = ['Q1 2023', 'Q2 2023', 'Q3 2023', 'Q4 2023', 'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024']

# Define Quarters for Sheet 2 (2025)
q_24_25 = ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025']

# Call the function to process both sheets
df_part1 = clean_excel_sheet(file_path, 'NetRevenues(22-24)', q_22_24)
df_part2 = clean_excel_sheet(file_path, 'NetRevenues(24-25)', q_24_25)

# Combine into a full dataset
full_dataset = pd.concat([df_part1, df_part2], ignore_index=True)

print("ETL Pipeline complete. Total rows:", len(full_dataset))
print(full_dataset.head(10))

ETL Pipeline complete. Total rows: 360
                          segment   period  revenue_eur_m
0                  Corporate Bank  Q1 2023            NaN
1     Corporate Treasury Services  Q1 2023    1191.557024
2   Institutional Client Services  Q1 2023     444.455003
3                Business Banking  Q1 2023     337.003158
4            Total Corporate Bank  Q1 2023    1973.015185
5                       of which:  Q1 2023            NaN
6             Net interest income  Q1 2023    1333.052912
7  Net commissions and fee income  Q1 2023     575.887964
8                Remaining income  Q1 2023      64.074310
9                 Investment Bank  Q1 2023            NaN


STEP2: Loading Data into SQLite

In [7]:
import sqlite3

In [9]:
connection = sqlite3.connect('deutsche_bank.db')
# 2. Write the DataFrame into a database table called 'net_revenues'
full_dataset.to_sql('net_revenues', connection, index=False, if_exists='replace')

print("Database table 'net_revenues' created successfully!")

# 3. Always close the connection when done with writing
connection.close()

Database table 'net_revenues' created successfully!


STEP3: Performing SQL Analytics to Address the Business Question in Hand

BUSINESS QUESTION1: What is the total net revenue generated by each main segment across the entire 12-quarter period (2023–2025), sorted from highest to lowest?

In [ ]:
connection = sqlite3.connect('deutsche_bank.db')
sql_query = """
SELECT 
    segment, 
    SUM(revenue_eur_m) AS total_revenue_eur_m
FROM net_revenues
WHERE segment IN (
    'Total Corporate Bank',
    'Total Investment Bank',
    'Total Private Bank',
    'Total Asset Management'
)
GROUP BY segment
ORDER BY total_revenue_eur_m DESC;
"""
df_summary = pd.read_sql(sql_query, connection)
print(df_summary)

connection.close()

BUSINESS QUESTION2: What are the quarterly revenue trend for the Investment Bank across all 12 quarters (from Q1 2023 to Q4 2025), along with a calculation showing its percentage share of total primary division revenue in each quarter.

In [4]:
connection = sqlite3.connect('deutsche_bank.db')
sql_query = """
SELECT 
    *
FROM net_revenues;
"""

database = pd.read_sql(sql_query, connection)
print(database)

connection.close()

                                               segment   period  revenue_eur_m
0                                       Corporate Bank  Q1 2023            NaN
1                          Corporate Treasury Services  Q1 2023    1191.557024
2                        Institutional Client Services  Q1 2023     444.455003
3                                     Business Banking  Q1 2023     337.003158
4                                 Total Corporate Bank  Q1 2023    1973.015185
..                                                 ...      ...            ...
355                                              Other  Q4 2025      41.092028
356                             Total Asset Management  Q4 2025     887.663221
357                                  Corporate & Other  Q4 2025      62.310466
358                                       Net revenues  Q4 2025    7725.615633
359  Footnotes and definitions of certain financial...  Q4 2025            NaN

[360 rows x 3 columns]


In [11]:
connection = sqlite3.connect('deutsche_bank.db')
sql_query = """
SELECT
    period,
    SUM(revenue_eur_m) AS total_net_revenue
FROM net_revenues
WHERE segment = 'Total Investment Bank'
GROUP BY period
ORDER BY
    SUBSTR(period, 4, 4) ASC, 
    SUBSTR(period, 1, 2) ASC;
"""

quaterly_revenue = pd.read_sql(sql_query, connection)
print(quaterly_revenue)

connection.close()

     period  total_net_revenue
0   Q1 2023        2691.308175
1   Q2 2023        2360.873127
2   Q3 2023        2270.942599
3   Q4 2023        1836.773201
4   Q1 2024        3046.629933
5   Q2 2024        2598.704993
6   Q3 2024        2522.665585
7   Q4 2024        2389.517628
8   Q1 2025        3362.258558
9   Q2 2025        2686.817170
10  Q3 2025        2977.968810
11  Q4 2025        2514.044711


BUSINESS QUESTION3: Determine the Q-O-Q and Y-O-Y Growth Trend

3.1---> Quarter-over-Quarter Growth: How did we perform compared to the previous quarter?
3.2---> Year-over-Year Growth: How did this quarter perform compared to the exact same quarter last year?

In [12]:
import sqlite3
import pandas as pd

connection = sqlite3.connect('deutsche_bank.db')

sql_variance = """
WITH ordered_quarters AS (
    SELECT 
        period,
        revenue_eur_m,
        SUBSTR(period, 4, 4) AS yr,
        SUBSTR(period, 1, 2) AS qtr
    FROM net_revenues
    WHERE segment = 'Total Investment Bank'
    ORDER BY yr ASC, qtr ASC
)
SELECT 
    period,
    ROUND(revenue_eur_m, 2) AS current_rev_m,
    
    -- 1. Previous Quarter Revenue
    ROUND(LAG(revenue_eur_m, 1) OVER (), 2) AS prev_qtr_rev_m,
    
    -- 2. Quarter-over-Quarter (QoQ) Growth %
    ROUND(((revenue_eur_m - LAG(revenue_eur_m, 1) OVER ()) / LAG(revenue_eur_m, 1) OVER ()) * 100, 2) AS qoq_growth_pct,
    
    -- 3. Same Quarter Previous Year Revenue
    ROUND(LAG(revenue_eur_m, 4) OVER (), 2) AS same_qtr_prev_yr_m,
    
    -- 4. Year-over-Year (YoY) Growth %
    ROUND(((revenue_eur_m - LAG(revenue_eur_m, 4) OVER ()) / LAG(revenue_eur_m, 4) OVER ()) * 100, 2) AS yoy_growth_pct

FROM ordered_quarters;
"""

df_variance = pd.read_sql(sql_variance, connection)
print(df_variance)

connection.close()

     period  current_rev_m  prev_qtr_rev_m  qoq_growth_pct  \
0   Q1 2023        2691.31             NaN             NaN   
1   Q2 2023        2360.87         2691.31          -12.28   
2   Q3 2023        2270.94         2360.87           -3.81   
3   Q4 2023        1836.77         2270.94          -19.12   
4   Q1 2024        3046.63         1836.77           65.87   
5   Q2 2024        2598.70         3046.63          -14.70   
6   Q3 2024        2522.67         2598.70           -2.93   
7   Q4 2024        2389.52         2522.67           -5.28   
8   Q1 2025        3362.26         2389.52           40.71   
9   Q2 2025        2686.82         3362.26          -20.09   
10  Q3 2025        2977.97         2686.82           10.84   
11  Q4 2025        2514.04         2977.97          -15.58   

    same_qtr_prev_yr_m  yoy_growth_pct  
0                  NaN             NaN  
1                  NaN             NaN  
2                  NaN             NaN  
3                  NaN       